# CartRank: two-tower recommender (Kaggle)

Trains the two-tower model on Amazon Reviews 2023 Office_Products (temporal leave-last-out
splits, built in the CartRank repo) and runs the ablation **ID-only vs ID + text embedding**
for the item tower, scored with full-ranking HR@10 / NDCG@10 over all items on the **dev** split,
with a cold-start slice (items with fewer than 5 train interactions).

Use a **GPU** session. Internet is not needed. Attach the Kaggle Dataset that holds
`data.npz` and `item_text_emb.npy`. Only train and dev data are included; the test split is
deliberately absent.

In [1]:
import glob, json, os, time
import numpy as np, torch

npz = glob.glob("/kaggle/input/datasets/abhibarnwal707/cartrank-twotower/data.npz", recursive=True)[0]
txt = glob.glob("/kaggle/input/datasets/abhibarnwal707/cartrank-twotower/item_text_emb.npy", recursive=True)[0]
data = dict(np.load(npz))
text_emb = np.load(txt)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device, torch.__version__, {k: v.shape for k, v in data.items() if hasattr(v, "shape") and v.ndim})
print("users", int(data["n_users"]), "items", int(data["n_items"]), "text", text_emb.shape)
assert text_emb.shape[0] == int(data["n_items"])

cuda 2.11.0+cu128 {'train_hist': (1130954, 20), 'train_target': (1130954,), 'dev_hist': (223308, 20), 'dev_target': (223308,), 'train_indptr': (223309,), 'train_indices': (1354262,), 'item_train_count': (77551,)}
users 223308 items 77551 text (77551, 384)


## Model code (identical to `src/cartrank/recsys/two_tower.py` in the repo)

In [2]:
%%writefile two_tower.py
"""Two-tower recommender: torch + numpy only, so this file can be pasted into a Kaggle notebook.

Item tower: ID embedding, optionally plus a projection of the item's text embedding (cold items).
User tower: masked mean of the (shared) item vectors of the user's recent items, then an MLP.
Loss: in-batch sampled softmax with log-q popularity correction. Evaluation: full ranking over
all items, history excluded, pessimistic ties (same rule as cartrank.eval.recsys_eval).
"""

import time

import numpy as np
import torch
import torch.nn.functional as F
from torch import nn

COLD_THRESHOLD = 5


class TwoTower(nn.Module):
    def __init__(
        self,
        n_items: int,
        text_emb: np.ndarray | None = None,
        dim: int = 128,
        hidden: int = 256,
        seed: int = 0,
    ):
        super().__init__()
        torch.manual_seed(seed)
        self.n_items = n_items
        self.id_emb = nn.Embedding(n_items + 1, dim, padding_idx=n_items)  # last row = padding
        nn.init.normal_(self.id_emb.weight, std=0.1)
        with torch.no_grad():
            self.id_emb.weight[n_items].zero_()
        self.has_text = text_emb is not None
        if self.has_text:
            text = torch.cat(
                [torch.as_tensor(text_emb, dtype=torch.float32), torch.zeros(1, text_emb.shape[1])]
            )
            self.register_buffer("text", text)
            self.text_proj = nn.Linear(text_emb.shape[1], dim, bias=False)
        self.user_mlp = nn.Sequential(nn.Linear(dim, hidden), nn.ReLU(), nn.Linear(hidden, dim))

    def item_raw(self, idx: torch.Tensor) -> torch.Tensor:
        v = self.id_emb(idx)
        if self.has_text:
            v = v + self.text_proj(self.text[idx])
        return v

    def item_vec(self, idx: torch.Tensor) -> torch.Tensor:
        return F.normalize(self.item_raw(idx), dim=-1)

    def user_vec(self, hist: torch.Tensor) -> torch.Tensor:
        """hist: (B, L) item indices, padded with n_items."""
        mask = (hist != self.n_items).float().unsqueeze(-1)
        mean = (self.item_raw(hist) * mask).sum(1) / mask.sum(1).clamp(min=1.0)
        return F.normalize(self.user_mlp(mean), dim=-1)

    @torch.no_grad()
    def all_item_vecs(self, chunk: int = 16384) -> torch.Tensor:
        dev = self.id_emb.weight.device
        out = [
            self.item_vec(torch.arange(s, min(s + chunk, self.n_items), device=dev))
            for s in range(0, self.n_items, chunk)
        ]
        return torch.cat(out)


def sampled_softmax_loss(u, v, target, log_q, tau):
    """In-batch softmax; logits[i, j] = u_i . v_j / tau - log q(target_j).

    Other rows holding the same target item are masked out (they are not negatives).
    """
    logits = (u @ v.T) / tau - log_q[target][None, :]
    same = target[:, None] == target[None, :]
    eye = torch.eye(len(target), dtype=torch.bool, device=u.device)
    logits = logits.masked_fill(same & ~eye, float("-inf"))
    return F.cross_entropy(logits, torch.arange(len(target), device=u.device))


def _history_mask(indptr: np.ndarray, indices: np.ndarray, users: np.ndarray):
    """Row/column positions of the history items of `users` (CSR gather, vectorised)."""
    lens = indptr[users + 1] - indptr[users]
    rows = np.repeat(np.arange(len(users)), lens)
    offsets = np.arange(lens.sum()) - np.repeat(np.cumsum(lens) - lens, lens)
    cols = indices[np.repeat(indptr[users], lens) + offsets]
    return rows, cols


@torch.no_grad()
def evaluate(
    model,
    hist,
    target,
    user_ids,
    indptr,
    indices,
    item_count,
    device,
    batch: int = 2048,
    k: int = 10,
) -> dict:
    """Full-ranking HR@k / NDCG@k over all items for every row of (hist, target).

    `user_ids` index the CSR history to exclude (train for dev). Cold slice = target items with
    fewer than COLD_THRESHOLD train interactions.
    """
    model.eval()
    V = model.all_item_vecs()
    ranks = []
    for s in range(0, len(target), batch):
        h = torch.as_tensor(hist[s : s + batch], dtype=torch.long, device=device)
        scores = model.user_vec(h) @ V.T
        rows, cols = _history_mask(indptr, indices, user_ids[s : s + batch])
        scores[torch.as_tensor(rows, device=device), torch.as_tensor(cols, device=device)] = float(
            "-inf"
        )
        t = torch.as_tensor(target[s : s + batch], dtype=torch.long, device=device)
        ts = scores.gather(1, t[:, None])
        ranks.append(((scores >= ts).sum(1) - 1).cpu().numpy())
    r = np.concatenate(ranks)

    def metrics(x: np.ndarray) -> dict:
        hit = x < k
        return {
            "hr@10": float(hit.mean()),
            "ndcg@10": float((hit / np.log2(x + 2)).mean()),
            "n_users": int(len(x)),
        }

    cold = item_count[target] < COLD_THRESHOLD
    return {"all": metrics(r), "cold": metrics(r[cold]) if cold.any() else None}


def train(
    data: dict,
    text_emb: np.ndarray | None,
    device: str,
    dim: int = 128,
    tau: float = 0.05,
    lr: float = 1e-3,
    batch: int = 2048,
    epochs: int = 20,
    patience: int = 3,
    seed: int = 0,
    log=print,
) -> tuple[nn.Module, list[dict]]:
    """Train on `data` (see package_kaggle.build_arrays), selecting the epoch by dev NDCG@10.

    Returns the best-epoch model and per-epoch records. Only train and dev data are used.
    """
    n_items = int(data["n_items"])
    model = TwoTower(n_items, text_emb, dim=dim, seed=seed).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
    count = data["item_train_count"].astype(np.float64)
    log_q = torch.as_tensor(
        np.log((count + 1) / (count.sum() + n_items)), dtype=torch.float32, device=device
    )
    X, y = data["train_hist"], data["train_target"]
    rng = np.random.default_rng(seed)
    dev_users = np.arange(len(data["dev_target"]))
    best, best_state, bad, records = -1.0, None, 0, []
    for epoch in range(1, epochs + 1):
        model.train()
        t0 = time.time()
        order = rng.permutation(len(y))
        losses = []
        for s in range(0, len(order) - batch + 1, batch):  # drop the ragged last batch
            idx = order[s : s + batch]
            h = torch.as_tensor(X[idx], dtype=torch.long, device=device)
            t = torch.as_tensor(y[idx], dtype=torch.long, device=device)
            loss = sampled_softmax_loss(model.user_vec(h), model.item_vec(t), t, log_q, tau)
            opt.zero_grad()
            loss.backward()
            opt.step()
            losses.append(loss.item())
        train_s = time.time() - t0
        res = evaluate(
            model,
            data["dev_hist"],
            data["dev_target"],
            dev_users,
            data["train_indptr"],
            data["train_indices"],
            data["item_train_count"],
            device,
        )
        rec = {"epoch": epoch, "loss": float(np.mean(losses)), "train_seconds": train_s, **res}
        records.append(rec)
        log(
            f"epoch {epoch}: loss {rec['loss']:.4f} dev NDCG@10 {res['all']['ndcg@10']:.4f} "
            f"HR@10 {res['all']['hr@10']:.4f} ({train_s:.0f}s)"
        )
        if res["all"]["ndcg@10"] > best:
            best, bad = res["all"]["ndcg@10"], 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= patience:
                break
    model.load_state_dict(best_state)
    return model, records

Writing two_tower.py


In [3]:
import two_tower as tt

## Runs
Four runs: item tower ID-only vs ID + text, each at temperature 0.05 and 0.1. The epoch is chosen
by dev NDCG@10 (patience 3). Edit `EPOCHS`/`DIM` here if you want a different budget.

In [4]:
EPOCHS, DIM, BATCH, LR = 20, 128, 2048, 1e-3
results, models = {}, {}
for variant, emb in (("id_only", None), ("id_text", text_emb)):
    for tau in (0.05, 0.1):
        name = f"{variant}_tau{tau}"
        print("=== ", name)
        t0 = time.time()
        model, records = tt.train(data, emb, device, dim=DIM, tau=tau, lr=LR, batch=BATCH,
                                  epochs=EPOCHS)
        best = max(records, key=lambda r: r["all"]["ndcg@10"])
        results[name] = {"variant": variant, "tau": tau, "dim": DIM, "best_epoch": best["epoch"],
                         "dev_all": best["all"], "dev_cold": best["cold"],
                         "train_seconds_total": time.time() - t0, "epochs": records}
        torch.save(model.state_dict(), f"/kaggle/working/two_tower_{name}.pt")
        json.dump(results, open("/kaggle/working/two_tower_results.json", "w"), indent=1)

===  id_only_tau0.05
epoch 1: loss 7.9458 dev NDCG@10 0.0076 HR@10 0.0139 (6s)
epoch 2: loss 7.2808 dev NDCG@10 0.0141 HR@10 0.0253 (5s)
epoch 3: loss 6.7634 dev NDCG@10 0.0192 HR@10 0.0331 (5s)
epoch 4: loss 6.2330 dev NDCG@10 0.0205 HR@10 0.0352 (5s)
epoch 5: loss 5.7143 dev NDCG@10 0.0203 HR@10 0.0343 (5s)
epoch 6: loss 5.2685 dev NDCG@10 0.0200 HR@10 0.0336 (5s)
epoch 7: loss 4.9135 dev NDCG@10 0.0192 HR@10 0.0322 (5s)
===  id_only_tau0.1
epoch 1: loss 7.6886 dev NDCG@10 0.0077 HR@10 0.0147 (5s)
epoch 2: loss 7.1118 dev NDCG@10 0.0150 HR@10 0.0272 (5s)
epoch 3: loss 6.5923 dev NDCG@10 0.0196 HR@10 0.0342 (5s)
epoch 4: loss 6.1937 dev NDCG@10 0.0216 HR@10 0.0373 (5s)
epoch 5: loss 5.9116 dev NDCG@10 0.0222 HR@10 0.0384 (5s)
epoch 6: loss 5.6971 dev NDCG@10 0.0228 HR@10 0.0392 (5s)
epoch 7: loss 5.5284 dev NDCG@10 0.0227 HR@10 0.0389 (5s)
epoch 8: loss 5.3922 dev NDCG@10 0.0228 HR@10 0.0392 (5s)
epoch 9: loss 5.2791 dev NDCG@10 0.0226 HR@10 0.0388 (5s)
===  id_text_tau0.05
epoch 1: l

In [5]:
print(f"{'run':24s} {'epoch':>5s} {'HR@10':>8s} {'NDCG@10':>8s} {'cold HR@10':>11s} {'cold NDCG@10':>13s}")
for n, r in results.items():
    c = r["dev_cold"] or {"hr@10": float("nan"), "ndcg@10": float("nan")}
    print(f"{n:24s} {r['best_epoch']:5d} {r['dev_all']['hr@10']:8.4f} {r['dev_all']['ndcg@10']:8.4f} "
          f"{c['hr@10']:11.4f} {c['ndcg@10']:13.4f}")

run                      epoch    HR@10  NDCG@10  cold HR@10  cold NDCG@10
id_only_tau0.05              4   0.0352   0.0205      0.0001        0.0001
id_only_tau0.1               6   0.0392   0.0228      0.0003        0.0002
id_text_tau0.05              4   0.0436   0.0253      0.0004        0.0002
id_text_tau0.1               7   0.0462   0.0267      0.0005        0.0002
